In [1]:
from snowflake.snowpark.functions import col,trim,split,lit
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
import sys 
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np
import datetime
import math 
from dateutil.relativedelta import relativedelta
from snowflake_utils import *

In [2]:
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

In [3]:
#Read the table
df = session.sql("""SELECT * FROM 
                 MOP_DATABASE.SOQ.DAILY_FORECASTING_DATA_FOR_MODELLING_TFT
                 WHERE PARENT_DEALER_CODE_MODEL_FAMILY IN (SELECT PARENT_DEALER_CODE_MODEL_FAMILY
                FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)""")

In [4]:
df.columns

['PARENT_DEALER_CODE',
 'MODEL_FAMILY',
 'MODEL_FAMILY_CODE',
 'CAL_DATE',
 'YEAR',
 'MONTH',
 'DAY_OF_THE_WEEK',
 'DAY_OF_THE_MONTH',
 'PARENT_DEALER_CODE_MODEL_FAMILY',
 'MODEL_NAME',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME',
 'DATASET_TYPE',
 'HARTALIK_TEEJ',
 'GANESH_CHATURTHI',
 'JANMASHTAMI',
 'VISHWAKARMA_PUJA',
 'KARWA_CHAUTH',
 'ONAM',
 'MARRIAGE_DAY',
 '"N-16"',
 '"N-15"',
 '"N-14"',
 '"N-13"',
 '"N-12"',
 '"N-11"',
 '"N-10"',
 '"N-9"',
 '"N-8"',
 '"N-7"',
 '"N-6"',
 '"N-5"',
 '"N-4"',
 '"N-3"',
 '"N-2"',
 '"N-1"',
 'N',
 '"N+1"',
 '"N+2"',
 '"N+3"',
 '"N+4"',
 '"N+5"',
 '"N+6"',
 '"N+7"',
 '"N+8"',
 '"N+9"',
 '"N+10"',
 '"D-3"',
 '"D-2"',
 '"D-1"',
 'D',
 '"D+1"',
 '"D+2"',
 '"D+3"',
 '"D+4"',
 '"D+5"',
 '"D+6"',
 'SERIES_INDEX',
 'NET_SALES']

In [5]:
snowflake_utils.shape_of_snowpark_df(df)

(8823040, 64)

In [6]:
# df.filter(F.year("CAL_DATE")==2026).select("CAL_DATE","SERIES_INDEX").show()

In [7]:
df.show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY"  |"MODEL_FAMILY_CODE"                 |"CAL_DATE"  |"YEAR"  |"MONTH"  |"DAY_OF_THE_WEEK"  |"DAY_OF_THE_MONTH"  |"PARENT_DEALER_CODE_MODEL_FAMILY"          |"MODEL_NA

In [8]:
df.select(F.max('SERIES_INDEX')).show()

---------------------------
|"MAX(""SERIES_INDEX"")"  |
---------------------------
|488                      |
---------------------------



In [9]:
#Drop the dataset type column
df = df.drop('DATASET_TYPE')

In [10]:
data = df.to_pandas()

In [11]:
target_col = "NET_SALES"
group_col = "PARENT_DEALER_CODE_MODEL_FAMILY"
time_col = "SERIES_INDEX"


#Static covariates
static_covariates = [
    "PARENT_DEALER_CODE",
    "MODEL_FAMILY",
    "MODEL_FAMILY_CODE",
    "MODEL_NAME",
    "BRAKE_TYPE",
    "IGNITION_TYPE",
    "WHEEL_TYPE",
    "COLOUR",
    "DEALER_CITY",
    "X_CITY_CATEGORY",
    "ZONAL_OFFICE_NAME",
]


#Known future covariates
calendar_covariates = [
    "YEAR",
    "MONTH",
    "DAY_OF_THE_WEEK",
    "DAY_OF_THE_MONTH",
]

festival_covariates = [
    "HARTALIK_TEEJ",
    "GANESH_CHATURTHI",
    "JANMASHTAMI",
    "VISHWAKARMA_PUJA",
    "KARWA_CHAUTH",
    "ONAM",
    "MARRIAGE_DAY",

    "N-16", "N-15", "N-14", "N-13",
    "N-12", "N-11", "N-10", "N-9",
    "N-8", "N-7", "N-6", "N-5",
    "N-4", "N-3", "N-2", "N-1",
    "N",
    "N+1", "N+2", "N+3", "N+4",
    "N+5", "N+6", "N+7", "N+8", "N+9",

    "D-3", "D-2", "D-1", "D",
    "D+1", "D+2", "D+3",
    "D+4", "D+5", "D+6",
]

future_covariates = calendar_covariates + festival_covariates

import pandas as pd
import numpy as np

from darts import TimeSeries
from darts.dataprocessing.transformers import (
    Scaler,
    StaticCovariatesTransformer,
)
from darts.models import TFTModel

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


In [12]:
data = data.sort_values(
    [
        "PARENT_DEALER_CODE_MODEL_FAMILY",
        "SERIES_INDEX",
    ]
).reset_index(drop=True)


In [13]:
weekday_mapping = {
    "MON": 0,
    "TUE": 1,
    "WED": 2,
    "THU": 3,
    "FRI": 4,
    "SAT": 5,
    "SUN": 6,
}

if data["DAY_OF_THE_WEEK"].dtype == "object":
    data["DAY_OF_THE_WEEK"] = (
        data["DAY_OF_THE_WEEK"]
        .str.upper()
        .map(weekday_mapping)
    )


In [14]:

if data[future_covariates].isna().any().any():
    null_counts = (
        data[future_covariates]
        .isna()
        .sum()
    )

    print(
        null_counts[null_counts > 0]
        .sort_values(ascending=False)
    )

    raise ValueError(
        "Null values found in known future covariates."
    )

In [15]:
historical_data = data[
    data["YEAR"].isin([2023, 2024, 2025])
].copy()

future_2026_data = data[
    data["YEAR"].eq(2026)
].copy()


In [16]:
if historical_data[target_col].isna().any():
    raise ValueError(
        "Historical NET_SALES contains missing values."
    )

if future_2026_data[target_col].notna().any():
    print(
        "Warning: Some 2026 NET_SALES values are populated. "
        "Ensure these are not placeholder zeroes."
    )


target_input_data = historical_data[
    [time_col, group_col, target_col]
    + static_covariates
].copy()


In [17]:

target_series = TimeSeries.from_group_dataframe(
    df=target_input_data,
    group_cols=group_col,
    time_col=time_col,
    value_cols=target_col,
    static_cols=static_covariates,
    freq=1,
    fill_missing_dates=False,
)

print("Number of series:", len(target_series))
print("Length of first series:", len(target_series[0]))
print("First series index:", target_series[0].time_index[:5])
print("Last series index:", target_series[0].time_index[-5:])
print("Static covariates:")
print(target_series[0].static_covariates)

training_covariate_data = historical_data[
    [time_col, group_col]
    + future_covariates
].copy()

all_covariate_data = data[
    [time_col, group_col]
    + future_covariates
].copy()

training_future_covariates = (
    TimeSeries.from_group_dataframe(
        df=training_covariate_data,
        group_cols=group_col,
        time_col=time_col,
        value_cols=future_covariates,
        freq=1,
        fill_missing_dates=False,
    )
)

full_future_covariates = (
    TimeSeries.from_group_dataframe(
        df=all_covariate_data,
        group_cols=group_col,
        time_col=time_col,
        value_cols=future_covariates,
        freq=1,
        fill_missing_dates=False,
    )
)

assert len(target_series) == len(training_future_covariates)
assert len(target_series) == len(full_future_covariates)

print(len(training_future_covariates[0]))  # 366
print(len(full_future_covariates[0]))      # 488


Number of series: 18080
Length of first series: 366
First series index: RangeIndex(start=1, stop=6, step=1, name='SERIES_INDEX')
Last series index: RangeIndex(start=362, stop=367, step=1, name='SERIES_INDEX')
Static covariates:
static_covariates          PARENT_DEALER_CODE_MODEL_FAMILY PARENT_DEALER_CODE  \
NET_SALES          10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE              10001   

static_covariates MODEL_FAMILY                 MODEL_FAMILY_CODE MODEL_NAME  \
NET_SALES              DESTINI  DESTINI<>DRUM<>SELF<>CAST<>WHITE    DESTINI   

static_covariates BRAKE_TYPE IGNITION_TYPE WHEEL_TYPE COLOUR DEALER_CITY  \
NET_SALES               DRUM          SELF       CAST  WHITE    AMRITSAR   

static_covariates X_CITY_CATEGORY     ZONAL_OFFICE_NAME  
NET_SALES                   URBAN  Zonal Office - North  
366
488


In [18]:
df.select(F.max("SERIES_INDEX")).show()

---------------------------
|"MAX(""SERIES_INDEX"")"  |
---------------------------
|488                      |
---------------------------



In [24]:
validation_start_index = 246

train_series = []
validation_series = []

for series in target_series:
    train_part = series.slice(
        series.start_time(),
        validation_start_index - 1,
    )

    validation_part = series.slice(
        validation_start_index,
        series.end_time(),
    )

    train_series.append(train_part)
    validation_series.append(validation_part)


train_covariates = []
validation_covariates = []

for covariate_series in training_future_covariates:
    train_covariates.append(
        covariate_series.slice(
            covariate_series.start_time(),
            validation_start_index - 1,
        )
    )

    validation_covariates.append(
        covariate_series
    )


In [53]:
len(validation_series)

18080

In [42]:
len(train_covariates)

18080

In [47]:
validation_series[0]

Empty DataFrame
Columns: [NET_SALES]
Index: []

shape: (0, 1, 1), freq: 1, size: 0.00 B

Static covariates:
    static_covariates          PARENT_DEALER_CODE_MODEL_FAMILY PARENT_DEALER_CODE MODEL_FAMILY                 MODEL_FAMILY_CODE MODEL_NAME  ... WHEEL_TYPE COLOUR DEALER_CITY X_CITY_CATEGORY     ZONAL_OFFICE_NAME
    NET_SALES          10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE              10001      DESTINI  DESTINI<>DRUM<>SELF<>CAST<>WHITE    DESTINI  ...       CAST  WHITE    AMRITSAR           URBAN  Zonal Office - North

In [44]:
len(validation_covariates)

18080

In [25]:

target_scaler = Scaler(
    global_fit=False,
    n_jobs=-1,
)

future_covariate_scaler = Scaler(
    global_fit=True,
    n_jobs=-1,
)

static_transformer = StaticCovariatesTransformer(
    n_jobs=-1,
)


scaled_train_series = (
    target_scaler.fit_transform(train_series)
)

print(f"Scaled_train_series_1 success!")

scaled_train_series = (
    static_transformer.fit_transform(
        scaled_train_series
    )
)

print(f"Scaled_train_series_2 success!")

scaled_validation_series = (
    target_scaler.transform(validation_series)
)

print(f"Scaled_validation_series_1 success!")


scaled_validation_series = (
    static_transformer.transform(
        scaled_validation_series
    )
)

print(f"Scaled_validation_series_2 success!")


scaled_train_covariates = (
    future_covariate_scaler.fit_transform(
        train_covariates
    )
)

print(f"scaled_train_covariates success!")


scaled_validation_covariates = (
    future_covariate_scaler.transform(
        validation_covariates
    )
)

print(f"scaled_validation_covariates success!")



Scaled_train_series_1 success!
Scaled_train_series_2 success!
Scaled_validation_series_1 success!
Scaled_validation_series_2 success!
scaled_train_covariates success!
scaled_validation_covariates success!


### Modelling starts

In [27]:
import os
from datetime import datetime
from pytorch_lightning.callbacks import ModelCheckpoint

now = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")


WORK_DIR = os.getcwd()
MODEL_NAME = f"daily_forecasting_tft_{now}"

MODEL_DIR = os.path.join(WORK_DIR, MODEL_NAME)
CHECKPOINT_DIR = os.path.join(MODEL_DIR, "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("MODEL_DIR:", MODEL_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

MODEL_DIR: c:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Modelling\daily_forecasting_tft_2026-07-13_18_13_05
CHECKPOINT_DIR: c:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Modelling\daily_forecasting_tft_2026-07-13_18_13_05\checkpoints


In [29]:
# INPUT_CHUNK_LENGTH = 122
# OUTPUT_CHUNK_LENGTH = 122

import torch

from pytorch_lightning.callbacks import EarlyStopping

# early_stopping = EarlyStopping(
#     monitor="val_loss",
#     patience=10,
#     min_delta=1e-4,
#     mode="min",
# )

# model = TFTModel(
#     input_chunk_length=INPUT_CHUNK_LENGTH,
# #     output_chunk_length=OUTPUT_CHUNK_LENGTH,

# #     hidden_size=32,
# #     lstm_layers=1,
# #     num_attention_heads=4,
# #     dropout=0.1,

#     batch_size=256,
#     n_epochs=100,

#     likelihood=None,
#     loss_fn=torch.nn.HuberLoss(),

#     random_state=42,

#     add_relative_index=True,

#     save_checkpoints=True,
#     force_reset=True,
#     model_name="daily_festive_tft_boss_approach",

#     pl_trainer_kwargs={
#         "accelerator": "gpu"
#             if torch.cuda.is_available()
#             else "cpu",
#         "devices": 1,
#         "callbacks": [early_stopping],
#         "gradient_clip_val": 0.1,
#     },
# )

import torch
from darts.models import TFTModel
from pytorch_lightning.callbacks import EarlyStopping

# Enable TF32 for matrix math speedups on modern NVIDIA cards
torch.set_float32_matmul_precision('high')

INPUT_CHUNK_LENGTH = 122
OUTPUT_CHUNK_LENGTH = 122

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    min_delta=1e-4,
    mode="min",
)

model = TFTModel(
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,

    hidden_size=32,
    lstm_layers=1,
    num_attention_heads=4,
    dropout=0.1,

    batch_size=256,
    n_epochs=100,

    likelihood=None,
    loss_fn=torch.nn.HuberLoss(),

    random_state=42,
    add_relative_index=True,

    save_checkpoints=True,
    force_reset=True,
    model_name="daily_festive_tft_boss_approach",
    
    # Keeps VSN projections fast and simple
    skip_interpolation=True,

    pl_trainer_kwargs={
        "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
        "devices": 1,
        "callbacks": [early_stopping],
        "gradient_clip_val": 0.1,
        "precision": "16-mixed",  # Leverages GPU Tensor Cores
    }
)

In [30]:
model.fit(
    series=scaled_train_series,
    future_covariates=scaled_train_covariates,

    val_series=scaled_validation_series,
    val_future_covariates=scaled_validation_covariates,
    dataloader_kwargs={
        "num_workers": 4,         # Parallellize data processing on CPU
        "pin_memory": True        # Fast page-locked VRAM transfers
    },

    verbose=True
)


ValueError: The input `series` are too short to extract even a single sample. Expected min length: `244`, received max length: `120`.


ValueError: The input `series` are too short to extract even a single sample. Expected min length: `244`, received max length: `120`.

In [38]:
scaled_forecasts = model.predict(
    n=122,
    series=scaled_train_series,
    future_covariates=scaled_validation_covariates,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [39]:
validation_forecasts = target_scaler.inverse_transform(
    scaled_forecasts
)

In [40]:
validation_forecasts = [
    forecast.map(
        lambda x: np.maximum(x, 0)
    )
    for forecast in validation_forecasts
]

In [61]:
len(validation_forecasts[0].values().flatten())

122

In [70]:
len(validation_forecasts)

18080

In [73]:
target_series[366].static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

'10041<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK SILVER'

In [75]:
# Build output DataFrame for Apr'26 to Jun'26 predictions
records = []

for i, (forecast, target_series_var) in enumerate(zip(validation_forecasts, target_series)):
    try:
        series_name = target_series_var.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]
    except (IndexError, AttributeError, KeyError) as e:
        print(f"Error at index {i}: {e}")
        break

    series_index    = forecast.time_index
    forecast_values = forecast.values().flatten()

    for index, pred in zip(series_index, forecast_values):
        records.append({
            'SERIES_INDEX'                   : index,
            'PARENT_DEALER_CODE_MODEL_FAMILY'  : series_name,
            'PREDICTED_SALES'                  : round(float(pred), 2)
        })

df_final_output = pd.DataFrame(records)
# df_final_output['MONTH_OF_SALE'] = pd.to_datetime(df_final_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
# df_final_output = df_final_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

# print(f'Output shape : {df_final_output.shape}')
# print(f'Months       : {df_final_output["MONTH_OF_SALE"].unique()}')
# print(f'Series count : {df_final_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')
df_final_output.tail(10)

,SERIES_INDEX,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES
2205750,356,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0
2205751,357,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0
2205752,358,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0
2205753,359,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0
2205754,360,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0
2205755,361,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0
2205756,362,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0
2205757,363,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0
2205758,364,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0
2205759,365,17116<>XPULSE<>DISC<>SELF<>SPOKE<>SILVER,0.0


In [76]:
df_final_output.head()

,SERIES_INDEX,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES
0,244,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.03
1,245,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.01
2,246,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.00
3,247,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.00
4,248,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.00


In [86]:
_2025_df = df.filter(F.year("CAL_DATE")==2025).select("PARENT_DEALER_CODE_MODEL_FAMILY","CAL_DATE","SERIES_INDEX","NET_SALES")
_2025_df.show()

----------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"         |"CAL_DATE"  |"SERIES_INDEX"  |"NET_SALES"  |
----------------------------------------------------------------------------------------
|10141<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE  |2025-09-01  |245             |0.000000     |
|10141<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE  |2025-09-02  |246             |0.000000     |
|10141<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE  |2025-09-03  |247             |0.000000     |
|10141<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE  |2025-09-04  |248             |0.000000     |
|10141<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE  |2025-09-05  |249             |0.000000     |
|10141<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE  |2025-09-06  |250             |0.000000     |
|10141<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE  |2025-09-07  |251             |0.000000     |
|10141<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE  |2025-09-08  |252             |0.000000     |
|10141<>SPLENDOR+<>DR

In [87]:
snowpark_output = session.create_dataframe(df_final_output)

In [88]:
snowpark_output.show()

--------------------------------------------------------------------------------
|"SERIES_INDEX"  |"PARENT_DEALER_CODE_MODEL_FAMILY"        |"PREDICTED_SALES"  |
--------------------------------------------------------------------------------
|244             |10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE  |0.03               |
|245             |10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE  |0.01               |
|246             |10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE  |0.0                |
|247             |10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE  |0.0                |
|248             |10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE  |0.0                |
|249             |10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE  |0.02               |
|250             |10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE  |0.0                |
|251             |10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE  |0.0                |
|252             |10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE  |0.0                |
|253             |10001<>DES

In [97]:
# snowpark_output_alias = snowpark_output.alias("prediction")
# _2025_df_alias = _2025_df.alias("actual")
joined_df = snowpark_output.join(_2025_df,on=(snowpark_output["PARENT_DEALER_CODE_MODEL_FAMILY"]==_2025_df["PARENT_DEALER_CODE_MODEL_FAMILY"])&(snowpark_output["SERIES_INDEX"]==_2025_df["SERIES_INDEX"]),how='inner')

In [ ]:

# # joined_df.show()

In [102]:
joined_df.columns

['L_0006_SERIES_INDEX',
 'L_0006_PARENT_DEALER_CODE_MODEL_FAMILY',
 'PREDICTED_SALES',
 'R_0007_PARENT_DEALER_CODE_MODEL_FAMILY',
 'CAL_DATE',
 'R_0007_SERIES_INDEX',
 'NET_SALES']

In [100]:
for old_col in joined_df.columns:
    new_col = old_col.replace('"','')

    joined_df = joined_df.rename(old_col,new_col)


In [103]:
joined_df = joined_df.rename({"L_0006_PARENT_DEALER_CODE_MODEL_FAMILY":"PARENT_DEALER_CODE_MODEL_FAMILY","NET_SALES":"ACTUAL_SALES"})
joined_df = joined_df.drop("L_0006_SERIES_INDEX","R_0007_PARENT_DEALER_CODE_MODEL_FAMILY","R_0007_SERIES_INDEX")

In [104]:
joined_df.show()

--------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"             |"PREDICTED_SALES"  |"CAL_DATE"  |"ACTUAL_SALES"  |
--------------------------------------------------------------------------------------------------
|11902<>PASSION<>DRUM<>SELF<>CAST<>SPORTS RED  |0.04               |2025-09-01  |0.000000        |
|11902<>PASSION<>DRUM<>SELF<>CAST<>SPORTS RED  |0.04               |2025-09-02  |0.000000        |
|11902<>PASSION<>DRUM<>SELF<>CAST<>SPORTS RED  |0.04               |2025-09-03  |0.000000        |
|11902<>PASSION<>DRUM<>SELF<>CAST<>SPORTS RED  |0.03               |2025-09-04  |0.000000        |
|11902<>PASSION<>DRUM<>SELF<>CAST<>SPORTS RED  |0.07               |2025-09-05  |0.000000        |
|11902<>PASSION<>DRUM<>SELF<>CAST<>SPORTS RED  |0.04               |2025-09-06  |0.000000        |
|11902<>PASSION<>DRUM<>SELF<>CAST<>SPORTS RED  |0.01               |2025-09-07  |0.000000        |
|11902<>PA

In [106]:
joined_df.to_pandas().to_csv(r"2025_daily_forecast_output.csv",index=False)

In [108]:
joined_df_pandas = joined_df.to_pandas()

In [107]:
from sklearn.metrics import mean_absolute_percentage_error,mean_absolute_error

In [112]:
mean_absolute_percentage_error(y_true=joined_df_pandas["ACTUAL_SALES"],y_pred=joined_df_pandas["PREDICTED_SALES"])

2238894207618563.5

In [ ]:
df.filter(F.year('CAL_DATE')==2025).select()

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE"  |"MODEL_FAMILY"  |"MODEL_FAMILY_CODE"                     |"CAL_DATE"  |"YEAR"  |"MONTH"  |"DAY_OF_THE_WEEK"  |"DAY_OF_THE_MONTH"  |"PARENT_DEALER_CODE_MODEL_FAMILY"              |